# OCR 쪽별 텍스트 점검
합성 PDF와 가짜 TSV 결과로 페이지 JSON 구조 및 통계를 확인합니다.

In [ ]:
from pathlib import Path
import hashlib
import pymupdf
from payment_card.file_pipeline.extraction.ocr_pages import create_ocr_trace

# 경로는 필요에 맞게 변경합니다.
sample_pdf = Path('/tmp/ocr-pages-synthetic.pdf')

doc = pymupdf.open()
doc.new_page()
doc.save(sample_pdf)
doc.close()

fake_tsv = ('level\tpage_num\tblock_num\tpar_num\tline_num\tword_num\tleft\ttop\twidth\theight\tconf\ttext\n'
            '5\t1\t1\t1\t1\t1\t0\t0\t5\t5\t95\t합성\n'
            '5\t1\t1\t1\t1\t2\t6\t0\t5\t5\t90\t12345\n'
            '5\t1\t1\t1\t1\t3\t12\t0\t5\t5\t88\t원\n')
trace = create_ocr_trace(sample_pdf, hashlib.sha256(sample_pdf.read_bytes()).hexdigest(),
                         run_tesseract=lambda *_: fake_tsv)

[{**{key: page[key] for key in ('page_no', 'chunk_status')}, **page['quality']}
 for page in trace['pages']]